In [43]:
import accountid
import pandas
import time
from datetime import datetime, timedelta, timezone

In [44]:
# Status confirmation making sure connection to the Alpaca API is established
status_code= accountid.response.status_code

if status_code== 200:
    print('Connected to Alpaca API successfully')
    account_info= accountid.response.json()
else:
    print('Connection failed')

Connected to Alpaca API successfully


In [45]:
# Fetches price data of the candle depending on whether the timeframe parameter is 1,5,10,15,or 30 mins. It then assigns the empty parameter dataframe with the live data from Alpaca API.

def get_recent_price_data(symbol="AAPL", timeframe="15Min", candle_count=100):
    end_time = datetime.now(timezone.utc)

    if timeframe == "15Min":
        minutes_per_candle = 15
    elif timeframe == "1Min":
        minutes_per_candle = 1
    elif timeframe == "5Min":
        minutes_per_candle = 5
    elif timeframe == "30Min":
        minutes_per_candle = 30
    else:
        minutes_per_candle = 15

    start_time = end_time - timedelta(minutes=minutes_per_candle * candle_count)

    params = {
        "symbols": symbol,
        "timeframe": timeframe,
        "start": start_time.isoformat().replace("+00:00", "Z"),
        "end": end_time.isoformat().replace("+00:00", "Z"),
        "limit": candle_count,
        "feed": "iex",
    }

    response = accountid.requests.get(
        "https://data.alpaca.markets/v2/stocks/bars",
        headers=accountid.HEADERS,
        params=params
    )

    print(response.status_code)

    data = response.json()
    bars = data.get("bars", {}).get(symbol, [])

    if not bars:
        raise ValueError(f"No bars found for {symbol}.")

    price_df = pandas.DataFrame(bars)

    return price_df

In [59]:
df = get_recent_price_data(
    symbol="SPY",
    timeframe="15Min",
    candle_count=100
)

display(df.tail())

200


,c,h,l,n,o,t,v,vw
12,756.34,757.36,756.34,4131,757.21,2026-05-29T19:45:00Z,379856,756.716879
13,756.52,756.64,756.34,16,756.34,2026-05-29T20:00:00Z,9517,756.530299
14,755.98,756.12,755.98,5,756.10,2026-05-29T20:15:00Z,418,756.033333
15,756.00,756.18,756.00,5,756.04,2026-05-29T20:30:00Z,663,756.096923
16,755.95,756.27,755.95,3,756.27,2026-05-29T20:45:00Z,520,756.083942


In [60]:
# Styling and formatting the columns of the pandas table as well as renaming the columns
def format_price_table(df):
    formatted_df = df.copy()

    formatted_df = formatted_df.rename(columns={
        "t": "Time",
        "o": "Open",
        "h": "High",
        "l": "Low",
        "c": "Close",
        "v": "Volume",
        "n": "Trade Count",
        "vw": "VWAP"
    })

    formatted_df["Time"] = pandas.to_datetime(
        formatted_df["Time"],
        utc=True
    )
    formatted_df["Time"] = formatted_df["Time"].dt.tz_convert(
        "America/New_York"
    )

    formatted_df = formatted_df[
        [
            "Time",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",
            "Trade Count",
            "VWAP"
        ]
    ]

    return formatted_df


In [61]:
formatted_df = format_price_table(df)
display(formatted_df.head())

,Time,Open,High,Low,Close,Volume,Trade Count,VWAP
0,2026-05-29 12:45:00-04:00,757.54,757.55,756.63,756.80,24109,611,757.034848
1,2026-05-29 13:00:00-04:00,756.85,757.02,756.54,756.78,26588,716,756.772500
2,2026-05-29 13:15:00-04:00,756.79,756.99,756.26,756.63,31395,802,756.602477
3,2026-05-29 13:30:00-04:00,756.65,756.67,756.02,756.09,25645,746,756.388404
4,2026-05-29 13:45:00-04:00,756.08,756.27,755.24,755.29,39016,973,755.838868


In [62]:
# Function to calculate indicators

def calculate_indicators(df):
    indicators_df = df.copy()

    # Make sure close prices are numeric
    indicators_df["c"] = pandas.to_numeric(indicators_df["c"], errors="coerce")
    indicators_df["v"] = pandas.to_numeric(indicators_df["v"], errors="coerce")

    # Simple Moving Averages
    indicators_df["SMA_20"] = indicators_df["c"].rolling(window=20).mean()
    indicators_df["SMA_50"] = indicators_df["c"].rolling(window=50).mean()

    # Exponential Moving Averages
    indicators_df["EMA_12"] = indicators_df["c"].ewm(span=12, adjust=False).mean()
    indicators_df["EMA_26"] = indicators_df["c"].ewm(span=26, adjust=False).mean()

    # MACD
    indicators_df["MACD"] = indicators_df["EMA_12"] - indicators_df["EMA_26"]
    indicators_df["MACD_Signal"] = indicators_df["MACD"].ewm(span=9, adjust=False).mean()
    indicators_df["MACD_Histogram"] = indicators_df["MACD"] - indicators_df["MACD_Signal"]

    # RSI
    delta = indicators_df["c"].diff()

    gains = delta.where(delta > 0, 0)
    losses = -delta.where(delta < 0, 0)

    avg_gain = gains.rolling(window=14).mean()
    avg_loss = losses.rolling(window=14).mean()

    rs = avg_gain / avg_loss
    indicators_df["RSI_14"] = 100 - (100 / (1 + rs))

    # Volume moving average
    indicators_df["Volume_SMA_20"] = indicators_df["v"].rolling(window=20).mean()

    #Average True Range
    high_low = indicators_df["h"] - indicators_df["l"]
    high_previous_close = (indicators_df["h"] - indicators_df["c"].shift(1)).abs()
    low_previous_close = (indicators_df["l"] - indicators_df["c"].shift(1)).abs()

    true_range = pandas.concat(
        [high_low, high_previous_close, low_previous_close],
        axis=1
    ).max(axis=1)

    indicators_df["ATR_14"] = true_range.rolling(window=14).mean()
    return indicators_df

In [63]:
import numpy as np

def fix_indicator_data(indicator_df):
    fixed_df = indicator_df.copy()

    numeric_columns = [
        "o",
        "h",
        "l",
        "c",
        "v",
        "n",
        "vw",
        "SMA_20",
        "SMA_50",
        "EMA_12",
        "EMA_26",
        "MACD",
        "MACD_Signal",
        "MACD_Histogram",
        "RSI_14",
        "Volume_SMA_20",
        "ATR_14"
    ]

    for column in numeric_columns:
        if column in fixed_df.columns:
            fixed_df[column] = pandas.to_numeric(fixed_df[column], errors="coerce")

    columns_to_fill_with_mean = [
        "RSI_14",
        "ATR_14",
        "SMA_20",
        "SMA_50",
        "Volume_SMA_20"
    ]

    for column in columns_to_fill_with_mean:
        if column in fixed_df.columns:
            if fixed_df[column].isna().all():
                fixed_df[column] = fixed_df[column].fillna(0.0)
            else:
                fixed_df[column] = fixed_df[column].fillna(fixed_df[column].mean())

    outlier_columns = [
        "v",
        "n"
    ]

    for column in outlier_columns:
        if column in fixed_df.columns:
            q1 = fixed_df[column].quantile(0.25)
            q3 = fixed_df[column].quantile(0.75)
            iqr = q3 - q1

            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            fixed_df[column] = fixed_df[column].clip(
                lower=lower_bound,
                upper=upper_bound
            )

    return fixed_df

In [64]:
indicator_df = calculate_indicators(df)
indicator_df = fix_indicator_data(indicator_df)
df_fixed = indicator_df

df_fixed.tail()

,c,h,l,n,o,t,v,vw,SMA_20,SMA_50,EMA_12,EMA_26,MACD,MACD_Signal,MACD_Histogram,RSI_14,Volume_SMA_20,ATR_14
12,756.34,757.36,756.34,1738.5,757.21,2026-05-29T19:45:00Z,96244,756.716879,0.0,0.0,756.449039,756.519185,-0.070146,-0.122898,0.052752,45.245243,0.0,0.762679
13,756.52,756.64,756.34,16.0,756.34,2026-05-29T20:00:00Z,9517,756.530299,0.0,0.0,756.459956,756.519246,-0.059289,-0.110176,0.050887,47.770701,0.0,0.800357
14,755.98,756.12,755.98,5.0,756.10,2026-05-29T20:15:00Z,418,756.033333,0.0,0.0,756.386117,756.479301,-0.093185,-0.106778,0.013593,43.988270,0.0,0.773214
15,756.00,756.18,756.00,5.0,756.04,2026-05-29T20:30:00Z,663,756.096923,0.0,0.0,756.326714,756.443798,-0.117084,-0.108839,-0.008244,44.281525,0.0,0.753214
16,755.95,756.27,755.95,3.0,756.27,2026-05-29T20:45:00Z,520,756.083942,0.0,0.0,756.268758,756.407220,-0.138462,-0.114764,-0.023698,44.940476,0.0,0.723929


In [65]:
#Formatting of the indicator table. Includes: time,open,high,low,close,volume,trade count,vwap,sma 20,sma 50,ema 12,ema 26,macd,macd signal,macd histogram,rsi 14,volume sma 20,atr 14

def format_indicator_table(indicator_df):
    pretty_indicator_df = indicator_df.rename(columns={
        "t": "Time",
        "o": "Open",
        "h": "High",
        "l": "Low",
        "c": "Close",
        "v": "Volume",
        "n": "Trade Count",
        "vw": "VWAP",
        "SMA_20": "SMA 20",
        "SMA_50": "SMA 50",
        "EMA_12": "EMA 12",
        "EMA_26": "EMA 26",
        "MACD_Signal": "MACD Signal",
        "MACD_Histogram": "MACD Histogram",
        "RSI_14": "RSI 14",
        "Volume_SMA_20": "Volume SMA 20",
        "ATR_14": "ATR 14"
    })

    pretty_indicator_df["Time"] = pandas.to_datetime(
        pretty_indicator_df["Time"],
        utc=True
    )
    pretty_indicator_df["Time"] = pretty_indicator_df["Time"].dt.tz_convert(
        "America/New_York"
    )
    pretty_indicator_df["Time"] = pretty_indicator_df["Time"].dt.strftime(
        "%Y-%m-%d %I:%M %p"
    )

    pretty_indicator_df = pretty_indicator_df[
        [
            "Time",
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",
            "Trade Count",
            "VWAP",
            "SMA 20",
            "SMA 50",
            "EMA 12",
            "EMA 26",
            "MACD",
            "MACD Signal",
            "MACD Histogram",
            "RSI 14",
            "Volume SMA 20",
            "ATR 14"
        ]
    ]

    return pretty_indicator_df

In [66]:
pretty_indicator_df = format_indicator_table(indicator_df)
display(pretty_indicator_df)

,Time,Open,High,Low,Close,Volume,Trade Count,VWAP,SMA 20,SMA 50,EMA 12,EMA 26,MACD,MACD Signal,MACD Histogram,RSI 14,Volume SMA 20,ATR 14
0,2026-05-29 12:45 PM,757.540,757.550,756.63,756.800,24109,611.0,757.034848,0.0,0.0,756.800000,756.800000,0.000000,0.000000,0.000000,45.245243,0.0,0.762679
1,2026-05-29 01:00 PM,756.850,757.020,756.54,756.780,26588,716.0,756.772500,0.0,0.0,756.796923,756.798519,-0.001595,-0.000319,-0.001276,45.245243,0.0,0.762679
2,2026-05-29 01:15 PM,756.790,756.990,756.26,756.630,31395,802.0,756.602477,0.0,0.0,756.771243,756.786036,-0.014793,-0.003214,-0.011579,45.245243,0.0,0.762679
3,2026-05-29 01:30 PM,756.650,756.670,756.02,756.090,25645,746.0,756.388404,0.0,0.0,756.666436,756.734477,-0.068041,-0.016179,-0.051862,45.245243,0.0,0.762679
4,2026-05-29 01:45 PM,756.080,756.270,755.24,755.290,39016,973.0,755.838868,0.0,0.0,756.454677,756.627479,-0.172802,-0.047504,-0.125298,45.245243,0.0,0.762679
5,2026-05-29 02:00 PM,755.300,756.125,755.14,755.755,61432,1065.0,755.673387,0.0,0.0,756.347034,756.562851,-0.215817,-0.081167,-0.134650,45.245243,0.0,0.762679
6,2026-05-29 02:15 PM,755.760,756.410,755.61,756.290,52963,1014.0,756.103020,0.0,0.0,756.338260,756.542640,-0.204380,-0.105809,-0.098571,45.245243,0.0,0.762679
7,2026-05-29 02:30 PM,756.300,756.755,756.30,756.430,49991,1016.0,756.581241,0.0,0.0,756.352374,756.534296,-0.181923,-0.121032,-0.060891,45.245243,0.0,0.762679
8,2026-05-29 02:45 PM,756.415,756.415,755.42,755.520,51121,1149.0,755.729209,0.0,0.0,756.224316,756.459163,-0.234847,-0.143795,-0.091052,45.245243,0.0,0.762679
9,2026-05-29 03:00 PM,755.495,756.500,755.36,756.360,57385,1062.0,756.005111,0.0,0.0,756.245191,756.451818,-0.206627,-0.156361,-0.050266,45.245243,0.0,0.762679


In [67]:
# Trading logic: This compares whether the higher or lower EMA crosses which will signal a bullish or bearish market, hence telling us when to sell or buy

def ema_crossover(df):
    if len(df) < 2:
        return "HOLD"

    latest_two_rows = df.dropna(subset=["EMA_12", "EMA_26"]).tail(2)

    if len(latest_two_rows) < 2:
        return "HOLD"

    previous_candle = latest_two_rows.iloc[0]
    current_candle = latest_two_rows.iloc[1]

    previous_ema_12 = previous_candle["EMA_12"]
    previous_ema_26 = previous_candle["EMA_26"]

    current_ema_12 = current_candle["EMA_12"]
    current_ema_26 = current_candle["EMA_26"]

    bullish_crossover = previous_ema_12 <= previous_ema_26 and current_ema_12 > current_ema_26
    bearish_crossover = previous_ema_12 >= previous_ema_26 and current_ema_12 < current_ema_26

    if bullish_crossover:
        return "BUY:EMA 12 crosses ABOVE EMA 26"

    if bearish_crossover:
        return "SELL:EMA 12 crosses BELOW EMA 26"

    return "HOLD"

#GLOBAL VARIABLE
trade_signal = ema_crossover(indicator_df)

print(f"EMA crossover signal: {trade_signal}")


EMA crossover signal: HOLD


In [68]:
# Creates the orders

def create_market_order(symbol, qty, side):
    order_data = {
        "symbol": symbol,
        "qty": qty,
        "side": side,
        "type": "market",
        "time_in_force": "day"
    }

    order_response = accountid.requests.post(
        "https://paper-api.alpaca.markets/v2/orders",
        headers=accountid.HEADERS,
        json=order_data
    )

    print(order_response.status_code)

    try:
        print(order_response.json())
    except Exception:
        print(order_response.text)

    return order_response

In [69]:
# Stop-Loss Mechanism

def calculate_trade_levels(indicator_df, tp_ratio=2):
    latest_candle = indicator_df.dropna(subset=["ATR_14"]).iloc[-1]

    entry_price = latest_candle["c"]

    stop_loss = entry_price - latest_candle["ATR_14"]

    stop_distance = entry_price - stop_loss

    take_profit = entry_price + (stop_distance * tp_ratio)

    trade_levels = {
        "entry_price": entry_price,
        "stop_loss": stop_loss,
        "stop_distance": stop_distance,
        "take_profit": take_profit,
        "tp_ratio": tp_ratio
    }

    return trade_levels

In [70]:
# Automation of the bot. The function is fed the trading signals which then will trigger the feeding of the trade levels (stop-loss). This will then execute the buy and sell orders. The automation process will check for current time vs last time to keep the process running.

def run_bot(
    symbol="AAPL",
    timeframe="15Min",
    candle_count=100,
    qty=1,
    place_order=False,
    tp_ratio=2,
    automate=False,
    max_runs=None,
    sleep_seconds=60,
    display_rows=20
):
    global df
    global formatted_df
    global indicator_df
    global pretty_indicator_df
    global trade_signal
    global order_response
    global trade_levels

    def execute_bot_once(run_number=None):
        global df
        global formatted_df
        global indicator_df
        global pretty_indicator_df
        global trade_signal
        global order_response
        global trade_levels

        if run_number is not None:
            print(f"Starting trading bot run #{run_number}...")
        else:
            print("Starting trading bot...")

        df = get_recent_price_data(
            symbol=symbol,
            timeframe=timeframe,
            candle_count=candle_count
        )

        if df is None or df.empty:
            raise ValueError(f"No price data found for {symbol}.")

        formatted_df = format_price_table(df)

        indicator_df = calculate_indicators(df)

        trade_signal = ema_crossover(indicator_df)

        pretty_indicator_df = format_indicator_table(indicator_df)

        order_response = None
        trade_levels = None

        print(f"Symbol: {symbol}")
        print(f"Timeframe: {timeframe}")
        print(f"Latest close: {df.iloc[-1]['c']}")
        print(f"EMA crossover signal: {trade_signal}")

        if trade_signal.startswith("BUY"):
            print("Buy signal detected")

            trade_levels = calculate_trade_levels(
                indicator_df,
                tp_ratio=tp_ratio
            )

            print(f"Entry price: {trade_levels['entry_price']:.2f}")
            print(f"Stop loss: {trade_levels['stop_loss']:.2f}")
            print(f"Stop distance: {trade_levels['stop_distance']:.2f}")
            print(f"Take profit: {trade_levels['take_profit']:.2f}")

            if place_order:
                order_response = create_market_order(symbol, qty, "buy")
            else:
                print("Order not placed because place_order=False")

        elif trade_signal.startswith("SELL"):
            print("Sell signal detected")

            if place_order:
                order_response = create_market_order(symbol, qty, "sell")
            else:
                print("Order not placed because place_order=False")

        else:
            print("No trade placed")

        display(pretty_indicator_df.tail(display_rows))

        return {
            "symbol": symbol,
            "timeframe": timeframe,
            "price_df": df,
            "formatted_df": formatted_df,
            "indicator_df": indicator_df,
            "pretty_indicator_df": pretty_indicator_df,
            "trade_signal": trade_signal,
            "trade_levels": trade_levels,
            "order_response": order_response
        }

    if not automate:
        return execute_bot_once()

    print("Bot automation started...")

    run_count = 0

    while True:
        run_count += 1

        try:
            bot_result = execute_bot_once(run_number=run_count)
        except Exception as error:
            print(f"Bot run failed: {error}")
            bot_result = None

        if max_runs is not None and run_count >= max_runs:
            print("Bot automation stopped.")
            return bot_result

        print(f"Waiting {sleep_seconds} seconds until next run...")
        time.sleep(sleep_seconds)

In [73]:
bot_result = run_bot(
    symbol="SPY",
    timeframe="15Min",
    candle_count=100,
    qty=1,
    place_order=True,
    tp_ratio=2,
    automate=True,
    max_runs=5,
    sleep_seconds=5
)

Bot automation started...
Starting trading bot run #1...
200
Symbol: SPY
Timeframe: 15Min
Latest close: 755.95
EMA crossover signal: HOLD
No trade placed


,Time,Open,High,Low,Close,Volume,Trade Count,VWAP,SMA 20,SMA 50,EMA 12,EMA 26,MACD,MACD Signal,MACD Histogram,RSI 14,Volume SMA 20,ATR 14
0,2026-05-29 12:45 PM,757.540,757.550,756.63,756.800,24109,611,757.034848,NaN,NaN,756.800000,756.800000,0.000000,0.000000,0.000000,NaN,NaN,NaN
1,2026-05-29 01:00 PM,756.850,757.020,756.54,756.780,26588,716,756.772500,NaN,NaN,756.796923,756.798519,-0.001595,-0.000319,-0.001276,NaN,NaN,NaN
2,2026-05-29 01:15 PM,756.790,756.990,756.26,756.630,31395,802,756.602477,NaN,NaN,756.771243,756.786036,-0.014793,-0.003214,-0.011579,NaN,NaN,NaN
3,2026-05-29 01:30 PM,756.650,756.670,756.02,756.090,25645,746,756.388404,NaN,NaN,756.666436,756.734477,-0.068041,-0.016179,-0.051862,NaN,NaN,NaN
4,2026-05-29 01:45 PM,756.080,756.270,755.24,755.290,39016,973,755.838868,NaN,NaN,756.454677,756.627479,-0.172802,-0.047504,-0.125298,NaN,NaN,NaN
5,2026-05-29 02:00 PM,755.300,756.125,755.14,755.755,61432,1065,755.673387,NaN,NaN,756.347034,756.562851,-0.215817,-0.081167,-0.134650,NaN,NaN,NaN
6,2026-05-29 02:15 PM,755.760,756.410,755.61,756.290,52963,1014,756.103020,NaN,NaN,756.338260,756.542640,-0.204380,-0.105809,-0.098571,NaN,NaN,NaN
7,2026-05-29 02:30 PM,756.300,756.755,756.30,756.430,49991,1016,756.581241,NaN,NaN,756.352374,756.534296,-0.181923,-0.121032,-0.060891,NaN,NaN,NaN
8,2026-05-29 02:45 PM,756.415,756.415,755.42,755.520,51121,1149,755.729209,NaN,NaN,756.224316,756.459163,-0.234847,-0.143795,-0.091052,NaN,NaN,NaN
9,2026-05-29 03:00 PM,755.495,756.500,755.36,756.360,57385,1062,756.005111,NaN,NaN,756.245191,756.451818,-0.206627,-0.156361,-0.050266,NaN,NaN,NaN


Waiting 5 seconds until next run...
Starting trading bot run #2...
200
Symbol: SPY
Timeframe: 15Min
Latest close: 755.95
EMA crossover signal: HOLD
No trade placed


,Time,Open,High,Low,Close,Volume,Trade Count,VWAP,SMA 20,SMA 50,EMA 12,EMA 26,MACD,MACD Signal,MACD Histogram,RSI 14,Volume SMA 20,ATR 14
0,2026-05-29 12:45 PM,757.540,757.550,756.63,756.800,24109,611,757.034848,NaN,NaN,756.800000,756.800000,0.000000,0.000000,0.000000,NaN,NaN,NaN
1,2026-05-29 01:00 PM,756.850,757.020,756.54,756.780,26588,716,756.772500,NaN,NaN,756.796923,756.798519,-0.001595,-0.000319,-0.001276,NaN,NaN,NaN
2,2026-05-29 01:15 PM,756.790,756.990,756.26,756.630,31395,802,756.602477,NaN,NaN,756.771243,756.786036,-0.014793,-0.003214,-0.011579,NaN,NaN,NaN
3,2026-05-29 01:30 PM,756.650,756.670,756.02,756.090,25645,746,756.388404,NaN,NaN,756.666436,756.734477,-0.068041,-0.016179,-0.051862,NaN,NaN,NaN
4,2026-05-29 01:45 PM,756.080,756.270,755.24,755.290,39016,973,755.838868,NaN,NaN,756.454677,756.627479,-0.172802,-0.047504,-0.125298,NaN,NaN,NaN
5,2026-05-29 02:00 PM,755.300,756.125,755.14,755.755,61432,1065,755.673387,NaN,NaN,756.347034,756.562851,-0.215817,-0.081167,-0.134650,NaN,NaN,NaN
6,2026-05-29 02:15 PM,755.760,756.410,755.61,756.290,52963,1014,756.103020,NaN,NaN,756.338260,756.542640,-0.204380,-0.105809,-0.098571,NaN,NaN,NaN
7,2026-05-29 02:30 PM,756.300,756.755,756.30,756.430,49991,1016,756.581241,NaN,NaN,756.352374,756.534296,-0.181923,-0.121032,-0.060891,NaN,NaN,NaN
8,2026-05-29 02:45 PM,756.415,756.415,755.42,755.520,51121,1149,755.729209,NaN,NaN,756.224316,756.459163,-0.234847,-0.143795,-0.091052,NaN,NaN,NaN
9,2026-05-29 03:00 PM,755.495,756.500,755.36,756.360,57385,1062,756.005111,NaN,NaN,756.245191,756.451818,-0.206627,-0.156361,-0.050266,NaN,NaN,NaN


Waiting 5 seconds until next run...
Starting trading bot run #3...
200
Symbol: SPY
Timeframe: 15Min
Latest close: 755.95
EMA crossover signal: HOLD
No trade placed


...

Waiting 5 seconds until next run...
Starting trading bot run #4...
200
Symbol: SPY
Timeframe: 15Min
Latest close: 755.95
EMA crossover signal: HOLD
No trade placed


,Time,Open,High,Low,Close,Volume,Trade Count,VWAP,SMA 20,SMA 50,EMA 12,EMA 26,MACD,MACD Signal,MACD Histogram,RSI 14,Volume SMA 20,ATR 14
0,2026-05-29 12:45 PM,757.540,757.550,756.63,756.800,24109,611,757.034848,NaN,NaN,756.800000,756.800000,0.000000,0.000000,0.000000,NaN,NaN,NaN
1,2026-05-29 01:00 PM,756.850,757.020,756.54,756.780,26588,716,756.772500,NaN,NaN,756.796923,756.798519,-0.001595,-0.000319,-0.001276,NaN,NaN,NaN
2,2026-05-29 01:15 PM,756.790,756.990,756.26,756.630,31395,802,756.602477,NaN,NaN,756.771243,756.786036,-0.014793,-0.003214,-0.011579,NaN,NaN,NaN
3,2026-05-29 01:30 PM,756.650,756.670,756.02,756.090,25645,746,756.388404,NaN,NaN,756.666436,756.734477,-0.068041,-0.016179,-0.051862,NaN,NaN,NaN
4,2026-05-29 01:45 PM,756.080,756.270,755.24,755.290,39016,973,755.838868,NaN,NaN,756.454677,756.627479,-0.172802,-0.047504,-0.125298,NaN,NaN,NaN
5,2026-05-29 02:00 PM,755.300,756.125,755.14,755.755,61432,1065,755.673387,NaN,NaN,756.347034,756.562851,-0.215817,-0.081167,-0.134650,NaN,NaN,NaN
6,2026-05-29 02:15 PM,755.760,756.410,755.61,756.290,52963,1014,756.103020,NaN,NaN,756.338260,756.542640,-0.204380,-0.105809,-0.098571,NaN,NaN,NaN
7,2026-05-29 02:30 PM,756.300,756.755,756.30,756.430,49991,1016,756.581241,NaN,NaN,756.352374,756.534296,-0.181923,-0.121032,-0.060891,NaN,NaN,NaN
8,2026-05-29 02:45 PM,756.415,756.415,755.42,755.520,51121,1149,755.729209,NaN,NaN,756.224316,756.459163,-0.234847,-0.143795,-0.091052,NaN,NaN,NaN
9,2026-05-29 03:00 PM,755.495,756.500,755.36,756.360,57385,1062,756.005111,NaN,NaN,756.245191,756.451818,-0.206627,-0.156361,-0.050266,NaN,NaN,NaN


Waiting 5 seconds until next run...
Starting trading bot run #5...
200
Symbol: SPY
Timeframe: 15Min
Latest close: 755.95
EMA crossover signal: HOLD
No trade placed


,Time,Open,High,Low,Close,Volume,Trade Count,VWAP,SMA 20,SMA 50,EMA 12,EMA 26,MACD,MACD Signal,MACD Histogram,RSI 14,Volume SMA 20,ATR 14
0,2026-05-29 12:45 PM,757.540,757.550,756.63,756.800,24109,611,757.034848,NaN,NaN,756.800000,756.800000,0.000000,0.000000,0.000000,NaN,NaN,NaN
1,2026-05-29 01:00 PM,756.850,757.020,756.54,756.780,26588,716,756.772500,NaN,NaN,756.796923,756.798519,-0.001595,-0.000319,-0.001276,NaN,NaN,NaN
2,2026-05-29 01:15 PM,756.790,756.990,756.26,756.630,31395,802,756.602477,NaN,NaN,756.771243,756.786036,-0.014793,-0.003214,-0.011579,NaN,NaN,NaN
3,2026-05-29 01:30 PM,756.650,756.670,756.02,756.090,25645,746,756.388404,NaN,NaN,756.666436,756.734477,-0.068041,-0.016179,-0.051862,NaN,NaN,NaN
4,2026-05-29 01:45 PM,756.080,756.270,755.24,755.290,39016,973,755.838868,NaN,NaN,756.454677,756.627479,-0.172802,-0.047504,-0.125298,NaN,NaN,NaN
5,2026-05-29 02:00 PM,755.300,756.125,755.14,755.755,61432,1065,755.673387,NaN,NaN,756.347034,756.562851,-0.215817,-0.081167,-0.134650,NaN,NaN,NaN
6,2026-05-29 02:15 PM,755.760,756.410,755.61,756.290,52963,1014,756.103020,NaN,NaN,756.338260,756.542640,-0.204380,-0.105809,-0.098571,NaN,NaN,NaN
7,2026-05-29 02:30 PM,756.300,756.755,756.30,756.430,49991,1016,756.581241,NaN,NaN,756.352374,756.534296,-0.181923,-0.121032,-0.060891,NaN,NaN,NaN
8,2026-05-29 02:45 PM,756.415,756.415,755.42,755.520,51121,1149,755.729209,NaN,NaN,756.224316,756.459163,-0.234847,-0.143795,-0.091052,NaN,NaN,NaN
9,2026-05-29 03:00 PM,755.495,756.500,755.36,756.360,57385,1062,756.005111,NaN,NaN,756.245191,756.451818,-0.206627,-0.156361,-0.050266,NaN,NaN,NaN


Bot automation stopped.
